In [ ]:
!pip install -q -U vllm
!pip install -q qwen-vl-utils==0.0.14
!pip install -q "bitsandbytes>=0.49.2"
!pip install -q python-Levenshtein   # unused but kept for pipeline parity, per Section 19 finding 3

In [ ]:
"""
Part B -- Zero-Shot Correction (RQ2)
Corrector: Qwen3-VL-8B-Instruct
Target: Tesseract's weakest-baseline predictions (Section 11, CER 0.7215)

Reuses the audited Qwen3-VL vLLM pipeline (Section 14, Section 19 audit) verbatim
for env setup / engine init / CER-WER functions / checkpointing pattern.
NEW for Part B: joins Tesseract's predictions into the input, and adds a
4-variant prompt builder (text-only vs text+image; with/without template context).

Run ONE config at a time by setting CONFIG_ID below, or loop all 4 in one session --
both patterns are included.
"""

import os, re, gc, time, ctypes
from pathlib import Path
import pandas as pd

# ============================================================
# ENV FIXES -- identical to qwen3vl_vllm.ipynb (Section 14 debugging history)
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# Preload libnvJitLink if needed (Section 14, fix #3) -- adjust path if your
# environment's CUDA lib location differs; harmless if already resolvable.
try:
    ctypes.CDLL("/usr/local/cuda/lib64/libnvJitLink.so.13", mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass  # already on linker path in this environment

# ============================================================
# CONFIG
# ============================================================
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
CONFIG_ID = 1   # 1=text-only/no-context, 2=text-only/with-context, 3=image/no-context, 4=image/with-context
DATASET_ROOT = Path("/kaggle/input/datasets/hy7haseeb/handwritten-dataset/kaggle_upload")
IMAGES_DIR   = DATASET_ROOT / "images"
EVAL_CSV     = DATASET_ROOT / "evaluation_set.csv"

TESSERACT_ROOT = Path("/kaggle/input/datasets/hy7haseeb/tesseract-real-prediction")
TESSERACT_CSV  = TESSERACT_ROOT / "tesseract_predictions.csv"

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)
PRED_CSV = OUT_DIR / f"qwen3vl_partB_config{CONFIG_ID}_predictions.csv"
SUMMARY_TXT = OUT_DIR / f"qwen3vl_partB_config{CONFIG_ID}_summary.txt"

BATCH_SIZE = 16
CHECKPOINT_EVERY_BATCHES = 5
MAX_MODEL_LEN = 4096       # same as Section 14/19
GPU_MEM_UTIL = 0.75        # same as Section 14
MAX_OUTPUT_TOKENS = 128    # same generation cap as Part A

TESSERACT_BASELINE_CER = 0.7215  # Section 11, for the comparison table at the end

# ============================================================
# CER / WER HELPERS -- identical to every other model in this project
# ============================================================
DIACRITIC_RE = re.compile(r"[\u064B-\u065F\u0670]")

def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    if len(gt) == 0: return 1.0 if len(pred) > 0 else 0.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    p, g = pred.split(), gt.split()
    if len(g) == 0: return 1.0 if len(p) > 0 else 0.0
    return edit_distance(p, g) / len(g)

# ============================================================
# LOAD EVAL SET + TESSERACT PREDICTIONS, JOIN ON file_id
# ============================================================
eval_df = pd.read_csv(EVAL_CSV)
tess_df = pd.read_csv(TESSERACT_CSV)[["file_id", "prediction"]].rename(
    columns={"prediction": "tesseract_prediction"}
)

df = eval_df.merge(tess_df, on="file_id", how="inner")
print(f"Eval set rows: {len(eval_df)} | Tesseract rows: {len(tess_df)} | Joined: {len(df)}")
if len(df) != len(eval_df):
    print(f"WARNING: {len(eval_df) - len(df)} rows dropped in the join -- check file_id alignment.")

df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")

# Resolve image paths (basename match, same as Section 19 finding 5)
disk_files = {p.name: p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}
df["resolved_path"] = df["image_path"].apply(lambda p: disk_files.get(str(p).replace("\\", "/").split("/")[-1]))
missing = df["resolved_path"].isna().sum()
print(f"Missing images: {missing}")
df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)

# Template-sentence pool (unique ground-truth sentences), for context configs 2 & 4
TEMPLATE_POOL = sorted(eval_df["text"].unique().tolist())
print(f"Template pool size: {len(TEMPLATE_POOL)} unique sentences")

# ============================================================
# 4-VARIANT PROMPT BUILDER
# ============================================================
BASE_INSTRUCTION = (
    "You are correcting the output of a low-quality OCR system that transcribed a single "
    "line of handwritten Urdu text in the Nastaliq script. The OCR system's guess is often "
    "wrong -- garbled, incomplete, or entirely incorrect. Your job is to produce the correct "
    "transcription. Output ONLY the corrected Urdu text -- no explanation, no commentary."
)

def build_system_prompt(config_id):
    if config_id in (2, 4):
        # with template-sentence context
        pool_text = "\n".join(TEMPLATE_POOL)
        return (
            BASE_INSTRUCTION
            + "\n\nThe correct transcription is guaranteed to be exactly one of the following "
              "known sentences. Choose the one that best matches the OCR guess"
            + (" and the image." if config_id == 4 else ".")
            + f"\n\nKnown sentences:\n{pool_text}"
        )
    else:
        return BASE_INSTRUCTION

def build_user_content(row, config_id):
    ocr_guess_text = f"OCR guess: {row['tesseract_prediction']}"
    if config_id in (3, 4):
        # text + image
        from PIL import Image
        img = Image.open(row["resolved_path"]).convert("RGB")
        return [
            {"type": "image", "image": img},
            {"type": "text", "text": ocr_guess_text},
        ]
    else:
        # text-only
        return [{"type": "text", "text": ocr_guess_text}]

SYSTEM_PROMPT = build_system_prompt(CONFIG_ID)
print(f"\n=== Config {CONFIG_ID} ===")
print(f"Image included: {CONFIG_ID in (3, 4)}")
print(f"Template context included: {CONFIG_ID in (2, 4)}")
if CONFIG_ID in (2, 4):
    print(f"System prompt length (chars): {len(SYSTEM_PROMPT)}")

# ============================================================
# vLLM ENGINE INIT -- same params as Section 14/19
# ============================================================
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    trust_remote_code=True,
    quantization="bitsandbytes",   # <-- this is what was missing
)

sampling_params = SamplingParams(
    temperature=0.0,
    repetition_penalty=1.05,   # same mild guard as Part A, Section 14
    max_tokens=MAX_OUTPUT_TOKENS,
)

# ============================================================
# RESUME LOGIC
# ============================================================
done_ids = set()
if PRED_CSV.exists():
    prior = pd.read_csv(PRED_CSV)
    done_ids = set(prior["file_id"].astype(str))
    print(f"Resuming: {len(done_ids)} already done for config {CONFIG_ID}")

remaining = df[~df["file_id"].astype(str).isin(done_ids)].reset_index(drop=True)
print(f"Remaining for config {CONFIG_ID}: {len(remaining)} / {len(df)}")

results = []
if PRED_CSV.exists():
    results = pd.read_csv(PRED_CSV).to_dict("records")

# ============================================================
# MAIN LOOP -- batched, same pattern as Section 14 (batch size 16)
# ============================================================
n_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

for b in range(n_batches):
    batch = remaining.iloc[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
    if len(batch) == 0:
        continue

    messages_batch = []
    for _, row in batch.iterrows():
        messages_batch.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_content(row, CONFIG_ID)},
        ])

    try:
        outputs = llm.chat(messages_batch, sampling_params)
        for row, out in zip(batch.itertuples(), outputs):
            pred_text = out.outputs[0].text.strip()
            gt = str(row.text)
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": pred_text,
                "cer_sensitive": cer(pred_text, gt, False),
                "cer_insensitive": cer(pred_text, gt, True),
                "wer_sensitive": wer(pred_text, gt, False),
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "error": "",
            })
    except Exception as e:
        # whole-batch exception -- sentinel rows, matching Section 19 finding 6's policy
        for row in batch.itertuples():
            gt = str(row.text)
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": "",
                "cer_sensitive": 1.0,
                "cer_insensitive": 1.0,
                "wer_sensitive": 1.0,
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "error": str(e)[:300],
            })

    if (b + 1) % CHECKPOINT_EVERY_BATCHES == 0 or b == n_batches - 1:
        pd.DataFrame(results).to_csv(PRED_CSV, index=False)
        print(f"Checkpoint: batch {b+1}/{n_batches}, {len(results)} rows saved")

pd.DataFrame(results).to_csv(PRED_CSV, index=False)

# ============================================================
# SUMMARY REPORT
# ============================================================
final = pd.DataFrame(results)
err_mask = final["error"].notna() & (final["error"] != "")
ok = final[~err_mask]

lines = []
lines.append("="*60)
lines.append(f"PART B -- QWEN3-VL-8B CORRECTOR -- CONFIG {CONFIG_ID}")
lines.append(f"Image included: {CONFIG_ID in (3,4)} | Template context: {CONFIG_ID in (2,4)}")
lines.append("="*60)
lines.append(f"Rows processed: {len(final)} | Errors: {err_mask.sum()}")
lines.append("")
lines.append(f"Tesseract baseline CER (Section 11):     {TESSERACT_BASELINE_CER:.4f}")
lines.append(f"Tesseract CER on this joined subset:      {ok['tesseract_cer_sensitive'].mean():.4f}")
lines.append(f"Corrected CER (config {CONFIG_ID}):                {ok['cer_sensitive'].mean():.4f}")
lines.append(f"Corrected WER (config {CONFIG_ID}):                {ok['wer_sensitive'].mean():.4f}")
delta = ok['tesseract_cer_sensitive'].mean() - ok['cer_sensitive'].mean()
lines.append(f"CER improvement (Tesseract - corrected):  {delta:+.4f}  ({'improved' if delta > 0 else 'worse'})")
lines.append(f"Rows where correction improved CER:        {(ok['cer_sensitive'] < ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Rows where correction made it worse:       {(ok['cer_sensitive'] > ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Rows unchanged:                             {(ok['cer_sensitive'] == ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Empty corrected predictions:                {(ok['corrected_prediction']=='').sum()}")
lines.append(f"Exact match to ground truth:                {100*(ok['corrected_prediction']==ok['ground_truth']).mean():.2f}%")

summary_text = "\n".join(lines)
print("\n" + summary_text)
SUMMARY_TXT.write_text(summary_text, encoding="utf-8")

print(f"\nSaved: {PRED_CSV}")
print(f"Saved: {SUMMARY_TXT}")
print(f"\nTo run the other 3 configs: change CONFIG_ID at the top and rerun this whole script.")

In [ ]:
"""
Part B -- Config 1 (Qwen3-VL corrector, text-only, no template context)
Qualitative error analysis -- same style as Sections 10, 11, 16, 20's manual
worst/best inspection, applied here to the correction task.
"""

import re
from pathlib import Path
import pandas as pd

PRED_CSV = Path("/kaggle/working/qwen3vl_partB_config1_predictions.csv")
OUT_TXT  = Path("/kaggle/working/qwen3vl_partB_config1_error_analysis.txt")

df = pd.read_csv(PRED_CSV)
df["ground_truth"] = df["ground_truth"].fillna("")
df["corrected_prediction"] = df["corrected_prediction"].fillna("")
df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")

# ============================================================
# 1. Exact-match diagnosis -- is 0% real, or a punctuation-stripping artifact?
# ============================================================
PUNCT_RE = re.compile(r"['\u2018\u2019\u061B\u060C\u06D4۔]")  # quotes, Urdu comma/semicolon/full-stop

def strip_punct(s):
    return PUNCT_RE.sub("", s).strip()

df["gt_stripped"] = df["ground_truth"].apply(strip_punct)
df["pred_stripped"] = df["corrected_prediction"].apply(strip_punct)
exact_raw = (df["corrected_prediction"] == df["ground_truth"]).sum()
exact_stripped = (df["pred_stripped"] == df["gt_stripped"]).sum()

# ============================================================
# 2. Worst / best examples (matching Section 10/11/16/20's format)
# ============================================================
worst10 = df.nlargest(10, "cer_sensitive")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction", "cer_sensitive", "tesseract_cer_sensitive"]
]
best10_nonperfect = df[df["cer_sensitive"] > 0].nsmallest(10, "cer_sensitive")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction", "cer_sensitive", "tesseract_cer_sensitive"]
]

# ============================================================
# 3. Where correction helped most / hurt most (delta vs Tesseract)
# ============================================================
df["delta"] = df["tesseract_cer_sensitive"] - df["cer_sensitive"]  # positive = improved
biggest_gains = df.nlargest(10, "delta")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction", "tesseract_cer_sensitive", "cer_sensitive", "delta"]
]
biggest_regressions = df.nsmallest(10, "delta")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction", "tesseract_cer_sensitive", "cer_sensitive", "delta"]
]

# ============================================================
# 4. Failure-mode categorization
# ============================================================
def categorize(row):
    pred, gt, tess = row["corrected_prediction"], row["ground_truth"], row["tesseract_prediction"]
    if pred == "":
        return "empty_output"
    if pred == tess:
        return "unchanged_copy_of_tesseract"
    if pred.strip() == gt.strip():
        return "exact_after_strip"
    words = pred.split()
    if len(words) >= 4 and len(set(words)) <= 2:
        return "repetition_loop"
    if row["cer_sensitive"] >= 1.5:
        return "fabrication_worse_than_tesseract"
    if row["delta"] > 0.1:
        return "meaningful_improvement"
    if row["delta"] < -0.1:
        return "meaningful_regression"
    return "marginal_change"

df["failure_category"] = df.apply(categorize, axis=1)
category_counts = df["failure_category"].value_counts()

# ============================================================
# 5. Per-writer breakdown (writers with >=5 samples), matching Section 10/11 style
# ============================================================
writer_stats = (
    df.groupby("writer_id")
    .agg(n=("cer_sensitive", "size"), mean_cer=("cer_sensitive", "mean"), mean_delta=("delta", "mean"))
    .query("n >= 5")
    .sort_values("mean_cer", ascending=False)
)

# ============================================================
# WRITE REPORT
# ============================================================
lines = []
lines.append("="*70)
lines.append("PART B -- CONFIG 1 (QWEN3-VL, TEXT-ONLY, NO TEMPLATE CONTEXT)")
lines.append("QUALITATIVE ERROR ANALYSIS")
lines.append("="*70)

lines.append("\n--- 1. Exact-match diagnosis ---")
lines.append(f"Exact match, raw strings:            {exact_raw} / {len(df)} ({100*exact_raw/len(df):.2f}%)")
lines.append(f"Exact match, punctuation-stripped:    {exact_stripped} / {len(df)} ({100*exact_stripped/len(df):.2f}%)")
if exact_stripped > exact_raw:
    lines.append("--> Some 'failures' are punctuation-only mismatches (quotes/full-stop), not content errors.")
else:
    lines.append("--> 0% exact match is NOT a punctuation artifact -- genuine content mismatches throughout.")

lines.append("\n--- 2. Failure-mode category counts ---")
for cat, n in category_counts.items():
    lines.append(f"  {cat:35s}: {n:5d} ({100*n/len(df):.1f}%)")

lines.append("\n--- 3. Worst 10 predictions (by corrected CER) ---")
for _, r in worst10.iterrows():
    lines.append(f"\n  [{r['file_id']}] corrected_CER={r['cer_sensitive']:.3f} | tesseract_CER={r['tesseract_cer_sensitive']:.3f}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 4. Best 10 non-perfect predictions ---")
for _, r in best10_nonperfect.iterrows():
    lines.append(f"\n  [{r['file_id']}] corrected_CER={r['cer_sensitive']:.3f} | tesseract_CER={r['tesseract_cer_sensitive']:.3f}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 5. Biggest 10 improvements over Tesseract (delta = tess_CER - corrected_CER) ---")
for _, r in biggest_gains.iterrows():
    lines.append(f"\n  [{r['file_id']}] delta={r['delta']:+.3f}  (tess={r['tesseract_cer_sensitive']:.3f} -> corrected={r['cer_sensitive']:.3f})")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 6. Biggest 10 regressions vs Tesseract ---")
for _, r in biggest_regressions.iterrows():
    lines.append(f"\n  [{r['file_id']}] delta={r['delta']:+.3f}  (tess={r['tesseract_cer_sensitive']:.3f} -> corrected={r['cer_sensitive']:.3f})")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 7. Per-writer CER (writers with >=5 samples, worst 10) ---")
lines.append(writer_stats.head(10).to_string())

report = "\n".join(lines)
print(report)
OUT_TXT.write_text(report, encoding="utf-8")
print(f"\n\nSaved full report to: {OUT_TXT}")

In [ ]:
"""
Part B -- Zero-Shot Correction (RQ2)
Corrector: Qwen3-VL-8B-Instruct
Target: Tesseract's weakest-baseline predictions (Section 11, CER 0.7215)

Reuses the audited Qwen3-VL vLLM pipeline (Section 14, Section 19 audit) verbatim
for env setup / engine init / CER-WER functions / checkpointing pattern.
NEW for Part B: joins Tesseract's predictions into the input, and adds a
4-variant prompt builder (text-only vs text+image; with/without template context).

Run ONE config at a time by setting CONFIG_ID below, or loop all 4 in one session --
both patterns are included.
"""

import os, re, gc, time, ctypes
from pathlib import Path
import pandas as pd

# ============================================================
# ENV FIXES -- identical to qwen3vl_vllm.ipynb (Section 14 debugging history)
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# Preload libnvJitLink if needed (Section 14, fix #3) -- adjust path if your
# environment's CUDA lib location differs; harmless if already resolvable.
try:
    ctypes.CDLL("/usr/local/cuda/lib64/libnvJitLink.so.13", mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass  # already on linker path in this environment

# ============================================================
# CONFIG
# ============================================================
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
CONFIG_ID = 3   # 1=text-only/no-context, 2=text-only/with-context, 3=image/no-context, 4=image/with-context
DATASET_ROOT = Path("/kaggle/input/datasets/hy7haseeb/handwritten-dataset/kaggle_upload")
IMAGES_DIR   = DATASET_ROOT / "images"
EVAL_CSV     = DATASET_ROOT / "evaluation_set.csv"

TESSERACT_ROOT = Path("/kaggle/input/datasets/hy7haseeb/tesseract-real-prediction")
TESSERACT_CSV  = TESSERACT_ROOT / "tesseract_predictions.csv"

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)
PRED_CSV = OUT_DIR / f"qwen3vl_partB_config{CONFIG_ID}_predictions.csv"
SUMMARY_TXT = OUT_DIR / f"qwen3vl_partB_config{CONFIG_ID}_summary.txt"

BATCH_SIZE = 16
CHECKPOINT_EVERY_BATCHES = 5
MAX_MODEL_LEN = 4096       # same as Section 14/19
GPU_MEM_UTIL = 0.75        # same as Section 14
MAX_OUTPUT_TOKENS = 128    # same generation cap as Part A

TESSERACT_BASELINE_CER = 0.7215  # Section 11, for the comparison table at the end

# ============================================================
# CER / WER HELPERS -- identical to every other model in this project
# ============================================================
DIACRITIC_RE = re.compile(r"[\u064B-\u065F\u0670]")

def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    if len(gt) == 0: return 1.0 if len(pred) > 0 else 0.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    p, g = pred.split(), gt.split()
    if len(g) == 0: return 1.0 if len(p) > 0 else 0.0
    return edit_distance(p, g) / len(g)

# ============================================================
# LOAD EVAL SET + TESSERACT PREDICTIONS, JOIN ON file_id
# ============================================================
eval_df = pd.read_csv(EVAL_CSV)
tess_df = pd.read_csv(TESSERACT_CSV)[["file_id", "prediction"]].rename(
    columns={"prediction": "tesseract_prediction"}
)

df = eval_df.merge(tess_df, on="file_id", how="inner")
print(f"Eval set rows: {len(eval_df)} | Tesseract rows: {len(tess_df)} | Joined: {len(df)}")
if len(df) != len(eval_df):
    print(f"WARNING: {len(eval_df) - len(df)} rows dropped in the join -- check file_id alignment.")

df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")

# Resolve image paths (basename match, same as Section 19 finding 5)
disk_files = {p.name: p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}
df["resolved_path"] = df["image_path"].apply(lambda p: disk_files.get(str(p).replace("\\", "/").split("/")[-1]))
missing = df["resolved_path"].isna().sum()
print(f"Missing images: {missing}")
df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)

# Template-sentence pool (unique ground-truth sentences), for context configs 2 & 4
TEMPLATE_POOL = sorted(eval_df["text"].unique().tolist())
print(f"Template pool size: {len(TEMPLATE_POOL)} unique sentences")

# ============================================================
# 4-VARIANT PROMPT BUILDER
# ============================================================
BASE_INSTRUCTION = (
    "You are correcting the output of a low-quality OCR system that transcribed a single "
    "line of handwritten Urdu text in the Nastaliq script. The OCR system's guess is often "
    "wrong -- garbled, incomplete, or entirely incorrect. Your job is to produce the correct "
    "transcription. Output ONLY the corrected Urdu text -- no explanation, no commentary."
)

def build_system_prompt(config_id):
    if config_id in (2, 4):
        # with template-sentence context
        pool_text = "\n".join(TEMPLATE_POOL)
        return (
            BASE_INSTRUCTION
            + "\n\nThe correct transcription is guaranteed to be exactly one of the following "
              "known sentences. Choose the one that best matches the OCR guess"
            + (" and the image." if config_id == 4 else ".")
            + f"\n\nKnown sentences:\n{pool_text}"
        )
    else:
        return BASE_INSTRUCTION

def build_user_content(row, config_id):
    ocr_guess_text = f"OCR guess: {row['tesseract_prediction']}"
    if config_id in (3, 4):
        # text + image
        from PIL import Image
        img = Image.open(row["resolved_path"]).convert("RGB")
        return [
            {"type": "image_pil", "image_pil": img},
            {"type": "text", "text": ocr_guess_text},
        ]
    else:
        # text-only
        return [{"type": "text", "text": ocr_guess_text}]

SYSTEM_PROMPT = build_system_prompt(CONFIG_ID)
print(f"\n=== Config {CONFIG_ID} ===")
print(f"Image included: {CONFIG_ID in (3, 4)}")
print(f"Template context included: {CONFIG_ID in (2, 4)}")
if CONFIG_ID in (2, 4):
    print(f"System prompt length (chars): {len(SYSTEM_PROMPT)}")

# ============================================================
# vLLM ENGINE INIT -- same params as Section 14/19
# ============================================================
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    trust_remote_code=True,
    quantization="bitsandbytes",   # <-- this is what was missing
)

sampling_params = SamplingParams(
    temperature=0.0,
    repetition_penalty=1.05,   # same mild guard as Part A, Section 14
    max_tokens=MAX_OUTPUT_TOKENS,
)

# ============================================================
# RESUME LOGIC
# ============================================================
done_ids = set()
if PRED_CSV.exists():
    prior = pd.read_csv(PRED_CSV)
    done_ids = set(prior["file_id"].astype(str))
    print(f"Resuming: {len(done_ids)} already done for config {CONFIG_ID}")

remaining = df[~df["file_id"].astype(str).isin(done_ids)].reset_index(drop=True)
print(f"Remaining for config {CONFIG_ID}: {len(remaining)} / {len(df)}")

results = []
if PRED_CSV.exists():
    results = pd.read_csv(PRED_CSV).to_dict("records")

# ============================================================
# MAIN LOOP -- batched, same pattern as Section 14 (batch size 16)
# ============================================================
n_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

for b in range(n_batches):
    batch = remaining.iloc[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
    if len(batch) == 0:
        continue

    messages_batch = []
    for _, row in batch.iterrows():
        messages_batch.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_content(row, CONFIG_ID)},
        ])

    try:
        outputs = llm.chat(messages_batch, sampling_params)
        for row, out in zip(batch.itertuples(), outputs):
            pred_text = out.outputs[0].text.strip()
            gt = str(row.text)
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": pred_text,
                "cer_sensitive": cer(pred_text, gt, False),
                "cer_insensitive": cer(pred_text, gt, True),
                "wer_sensitive": wer(pred_text, gt, False),
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "error": "",
            })
    except Exception as e:
        # whole-batch exception -- sentinel rows, matching Section 19 finding 6's policy
        for row in batch.itertuples():
            gt = str(row.text)
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": "",
                "cer_sensitive": 1.0,
                "cer_insensitive": 1.0,
                "wer_sensitive": 1.0,
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "error": str(e)[:300],
            })

    if (b + 1) % CHECKPOINT_EVERY_BATCHES == 0 or b == n_batches - 1:
        pd.DataFrame(results).to_csv(PRED_CSV, index=False)
        print(f"Checkpoint: batch {b+1}/{n_batches}, {len(results)} rows saved")

pd.DataFrame(results).to_csv(PRED_CSV, index=False)

# ============================================================
# SUMMARY REPORT
# ============================================================
final = pd.DataFrame(results)
err_mask = final["error"].notna() & (final["error"] != "")
ok = final[~err_mask]

lines = []
lines.append("="*60)
lines.append(f"PART B -- QWEN3-VL-8B CORRECTOR -- CONFIG {CONFIG_ID}")
lines.append(f"Image included: {CONFIG_ID in (3,4)} | Template context: {CONFIG_ID in (2,4)}")
lines.append("="*60)
lines.append(f"Rows processed: {len(final)} | Errors: {err_mask.sum()}")
lines.append("")
lines.append(f"Tesseract baseline CER (Section 11):     {TESSERACT_BASELINE_CER:.4f}")
lines.append(f"Tesseract CER on this joined subset:      {ok['tesseract_cer_sensitive'].mean():.4f}")
lines.append(f"Corrected CER (config {CONFIG_ID}):                {ok['cer_sensitive'].mean():.4f}")
lines.append(f"Corrected WER (config {CONFIG_ID}):                {ok['wer_sensitive'].mean():.4f}")
delta = ok['tesseract_cer_sensitive'].mean() - ok['cer_sensitive'].mean()
lines.append(f"CER improvement (Tesseract - corrected):  {delta:+.4f}  ({'improved' if delta > 0 else 'worse'})")
lines.append(f"Rows where correction improved CER:        {(ok['cer_sensitive'] < ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Rows where correction made it worse:       {(ok['cer_sensitive'] > ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Rows unchanged:                             {(ok['cer_sensitive'] == ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Empty corrected predictions:                {(ok['corrected_prediction']=='').sum()}")
lines.append(f"Exact match to ground truth:                {100*(ok['corrected_prediction']==ok['ground_truth']).mean():.2f}%")

summary_text = "\n".join(lines)
print("\n" + summary_text)
SUMMARY_TXT.write_text(summary_text, encoding="utf-8")

print(f"\nSaved: {PRED_CSV}")
print(f"Saved: {SUMMARY_TXT}")
print(f"\nTo run the other 3 configs: change CONFIG_ID at the top and rerun this whole script.")

# QWEN 2.5


In [ ]:
!pip install -q -U vllm
!pip install -q qwen-vl-utils==0.0.14
!pip install -q "bitsandbytes>=0.49.2"
!pip install -q python-Levenshtein   # unused but kept for pipeline parity, per Section 19 finding 3

In [2]:
"""
Part B -- Zero-Shot Correction (RQ2)
Corrector: Qwen2.5-VL-7B-Instruct
Config: 3 ONLY (image + text, no template-sentence context)
Target: Tesseract's weakest-baseline predictions (Section 11, CER 0.7215)

This is a clone of qwen3vl_partB_config{N}.py, model swapped to Qwen2.5-VL-7B-Instruct,
per the project's established clone-verbatim policy (Section 19/20 -- same pipeline,
model ID change only, so results stay comparable across correctors).

SCOPE NOTE (per project scoping decision): only Config 3 is run for this corrector.
Config 1 (text-only) is not re-run here -- that isolation was already established on
Qwen3-VL (Sections 26/27); this run tests generalization of the Config-3 (image-grounded)
effect to a second, weaker-Part-A-performing corrector. CONFIG_ID is left as a variable
(rather than deleted) for traceability against the original 4-variant design, but is
hardcoded to 3 -- do not repurpose this script to sweep other configs without updating
this note.

One deliberate, disclosed deviation from a pure clone, carried over from Qwen2.5-VL's
own Part A notebook (Section 20): max_num_batched_tokens is set explicitly rather than
left at vLLM's default, using the per-image token count already measured for this model
in Section 20 (708 tokens/image), following the same derivation
(per_image_tokens x batch_size x 1.25 margin) that prevented the InternVL-style
scheduling bottleneck documented in Section 17. This is not new per-model accuracy
tuning -- it is the same category of pre-flight check the project already applies to
every model, now correctly carried into Part B for the one model where it was originally
established as necessary.
"""

import os, re, gc, time, ctypes
from pathlib import Path
import pandas as pd

# ============================================================
# ENV FIXES -- identical to qwen25vl_vllm.ipynb / Qwen3-VL Part B script
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

try:
    ctypes.CDLL("/usr/local/cuda/lib64/libnvJitLink.so.13", mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass  # already on linker path in this environment

# ============================================================
# CONFIG
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
CONFIG_ID = 3   # HARDCODED -- only Config 3 is in scope for this corrector, see scope note above
DATASET_ROOT = Path("/kaggle/input/datasets/hy7haseeb/handwritten-dataset/kaggle_upload")
IMAGES_DIR   = DATASET_ROOT / "images"
EVAL_CSV     = DATASET_ROOT / "evaluation_set.csv"

TESSERACT_ROOT = Path("/kaggle/input/datasets/hy7haseeb/tesseract-real-prediction")
TESSERACT_CSV  = TESSERACT_ROOT / "tesseract_predictions.csv"

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)
PRED_CSV = OUT_DIR / f"qwen25vl_partB_config{CONFIG_ID}_predictions.csv"
SUMMARY_TXT = OUT_DIR / f"qwen25vl_partB_config{CONFIG_ID}_summary.txt"

BATCH_SIZE = 16
CHECKPOINT_EVERY_BATCHES = 5
MAX_MODEL_LEN = 4096       # same as Section 14/19/20
GPU_MEM_UTIL = 0.75        # same as Section 14/20

# Section 20's Cell 4b finding: Qwen2.5-VL measured ~708 tokens/image (vs. Qwen3-VL's
# assumed ~128). Deriving max_num_batched_tokens the same way rather than reusing
# Qwen3-VL's hardcoded 2048, to avoid the InternVL-style scheduling bottleneck (Section 17).
PER_IMAGE_TOKENS_MEASURED = 708
MAX_NUM_BATCHED_TOKENS = int(PER_IMAGE_TOKENS_MEASURED * BATCH_SIZE * 1.25)  # = 14,160

MAX_OUTPUT_TOKENS = 128    # same generation cap as Part A / Config 1

TESSERACT_BASELINE_CER = 0.7215  # Section 11, for the comparison table at the end

# ============================================================
# CER / WER HELPERS -- identical to every other model in this project
# ============================================================
DIACRITIC_RE = re.compile(r"[\u064B-\u065F\u0670]")

def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    if len(gt) == 0: return 1.0 if len(pred) > 0 else 0.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    p, g = pred.split(), gt.split()
    if len(g) == 0: return 1.0 if len(p) > 0 else 0.0
    return edit_distance(p, g) / len(g)

# ============================================================
# LOAD EVAL SET + TESSERACT PREDICTIONS, JOIN ON file_id
# ============================================================
eval_df = pd.read_csv(EVAL_CSV)
tess_df = pd.read_csv(TESSERACT_CSV)[["file_id", "prediction"]].rename(
    columns={"prediction": "tesseract_prediction"}
)

df = eval_df.merge(tess_df, on="file_id", how="inner")
print(f"Eval set rows: {len(eval_df)} | Tesseract rows: {len(tess_df)} | Joined: {len(df)}")
if len(df) != len(eval_df):
    print(f"WARNING: {len(eval_df) - len(df)} rows dropped in the join -- check file_id alignment.")

df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")

# Resolve image paths (basename match, same as Section 19 finding 5)
disk_files = {p.name: p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}
df["resolved_path"] = df["image_path"].apply(lambda p: disk_files.get(str(p).replace("\\", "/").split("/")[-1]))
missing = df["resolved_path"].isna().sum()
print(f"Missing images: {missing}")
df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)

# Template-sentence pool -- loaded for parity with the 4-variant builder, but unused
# since CONFIG_ID is hardcoded to 3 (no template context in this config).
TEMPLATE_POOL = sorted(eval_df["text"].unique().tolist())
print(f"Template pool size: {len(TEMPLATE_POOL)} unique sentences (unused in Config 3)")

# ============================================================
# 4-VARIANT PROMPT BUILDER -- kept intact for traceability against the original
# design; only the CONFIG_ID == 3 branch is ever exercised in this script.
# ============================================================
BASE_INSTRUCTION = (
    "You are correcting the output of a low-quality OCR system that transcribed a single "
    "line of handwritten Urdu text in the Nastaliq script. The OCR system's guess is often "
    "wrong -- garbled, incomplete, or entirely incorrect. Your job is to produce the correct "
    "transcription. Output ONLY the corrected Urdu text -- no explanation, no commentary."
)

def build_system_prompt(config_id):
    if config_id in (2, 4):
        pool_text = "\n".join(TEMPLATE_POOL)
        return (
            BASE_INSTRUCTION
            + "\n\nThe correct transcription is guaranteed to be exactly one of the following "
              "known sentences. Choose the one that best matches the OCR guess"
            + (" and the image." if config_id == 4 else ".")
            + f"\n\nKnown sentences:\n{pool_text}"
        )
    else:
        return BASE_INSTRUCTION

def build_user_content(row, config_id):
    ocr_guess_text = f"OCR guess: {row['tesseract_prediction']}"
    if config_id in (3, 4):
        from PIL import Image
        img = Image.open(row["resolved_path"]).convert("RGB")
        return [
            {"type": "image_pil", "image_pil": img},
            {"type": "text", "text": ocr_guess_text},
        ]
    else:
        return [{"type": "text", "text": ocr_guess_text}]

SYSTEM_PROMPT = build_system_prompt(CONFIG_ID)
print(f"\n=== Config {CONFIG_ID} ===")
print(f"Image included: {CONFIG_ID in (3, 4)}")
print(f"Template context included: {CONFIG_ID in (2, 4)}")

# ============================================================
# vLLM ENGINE INIT
# NOTE: {"type": "image", "image": img} content-part format matches Config 1's
# script as provided. If this vLLM version rejects it the way Section 27 found
# for vLLM 0.26.0 ("Unsupported chat content part type: 'image'"), switch to
# {"type": "image_pil", "image_pil": img} in build_user_content() above --
# check the very first batch's log output before trusting a long unattended run.
# ============================================================
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    trust_remote_code=True,
    quantization="bitsandbytes",
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,  # Qwen2.5-VL-specific, see note above
)

sampling_params = SamplingParams(
    temperature=0.0,
    repetition_penalty=1.05,   # same mild guard as Part A / Config 1
    max_tokens=MAX_OUTPUT_TOKENS,
)

# ============================================================
# RESUME LOGIC
# ============================================================
done_ids = set()
if PRED_CSV.exists():
    prior = pd.read_csv(PRED_CSV)
    done_ids = set(prior["file_id"].astype(str))
    print(f"Resuming: {len(done_ids)} already done for config {CONFIG_ID}")

remaining = df[~df["file_id"].astype(str).isin(done_ids)].reset_index(drop=True)
print(f"Remaining for config {CONFIG_ID}: {len(remaining)} / {len(df)}")

results = []
if PRED_CSV.exists():
    results = pd.read_csv(PRED_CSV).to_dict("records")

# ============================================================
# MAIN LOOP -- batched, same pattern as Config 1 script (batch size 16)
# ============================================================
n_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

for b in range(n_batches):
    batch = remaining.iloc[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
    if len(batch) == 0:
        continue

    messages_batch = []
    for _, row in batch.iterrows():
        messages_batch.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_content(row, CONFIG_ID)},
        ])

    try:
        outputs = llm.chat(messages_batch, sampling_params)
        for row, out in zip(batch.itertuples(), outputs):
            pred_text = out.outputs[0].text.strip()
            gt = str(row.text)
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": pred_text,
                "cer_sensitive": cer(pred_text, gt, False),
                "cer_insensitive": cer(pred_text, gt, True),
                "wer_sensitive": wer(pred_text, gt, False),
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "error": "",
            })
    except Exception as e:
        for row in batch.itertuples():
            gt = str(row.text)
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": "",
                "cer_sensitive": 1.0,
                "cer_insensitive": 1.0,
                "wer_sensitive": 1.0,
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "error": str(e)[:300],
            })

    if (b + 1) % CHECKPOINT_EVERY_BATCHES == 0 or b == n_batches - 1:
        pd.DataFrame(results).to_csv(PRED_CSV, index=False)
        print(f"Checkpoint: batch {b+1}/{n_batches}, {len(results)} rows saved")

pd.DataFrame(results).to_csv(PRED_CSV, index=False)

# ============================================================
# SUMMARY REPORT
# ============================================================
final = pd.DataFrame(results)
err_mask = final["error"].notna() & (final["error"] != "")
ok = final[~err_mask]

lines = []
lines.append("="*60)
lines.append(f"PART B -- QWEN2.5-VL-7B CORRECTOR -- CONFIG {CONFIG_ID}")
lines.append(f"Image included: {CONFIG_ID in (3,4)} | Template context: {CONFIG_ID in (2,4)}")
lines.append("="*60)
lines.append(f"Rows processed: {len(final)} | Errors: {err_mask.sum()}")
lines.append("")
lines.append(f"Tesseract baseline CER (Section 11):     {TESSERACT_BASELINE_CER:.4f}")
lines.append(f"Tesseract CER on this joined subset:      {ok['tesseract_cer_sensitive'].mean():.4f}")
lines.append(f"Corrected CER (config {CONFIG_ID}):                {ok['cer_sensitive'].mean():.4f}")
lines.append(f"Corrected WER (config {CONFIG_ID}):                {ok['wer_sensitive'].mean():.4f}")
delta = ok['tesseract_cer_sensitive'].mean() - ok['cer_sensitive'].mean()
lines.append(f"CER improvement (Tesseract - corrected):  {delta:+.4f}  ({'improved' if delta > 0 else 'worse'})")
lines.append(f"Rows where correction improved CER:        {(ok['cer_sensitive'] < ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Rows where correction made it worse:       {(ok['cer_sensitive'] > ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Rows unchanged:                             {(ok['cer_sensitive'] == ok['tesseract_cer_sensitive']).sum()} / {len(ok)}")
lines.append(f"Empty corrected predictions:                {(ok['corrected_prediction']=='').sum()}")
lines.append(f"Exact match to ground truth:                {100*(ok['corrected_prediction']==ok['ground_truth']).mean():.2f}%")

summary_text = "\n".join(lines)
print("\n" + summary_text)
SUMMARY_TXT.write_text(summary_text, encoding="utf-8")

print(f"\nSaved: {PRED_CSV}")
print(f"Saved: {SUMMARY_TXT}")
print(f"\nThis corrector is scoped to Config 3 only for this project -- see scope note")
print(f"at the top of this file before rerunning with a different CONFIG_ID.")

Eval set rows: 2465 | Tesseract rows: 2465 | Joined: 2465
Missing images: 0
Template pool size: 837 unique sentences (unused in Config 3)

=== Config 3 ===
Image included: True
Template context included: False
INFO 07-28 05:47:47 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.75, 'max_num_batched_tokens': 14160, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen2.5-VL-7B-Instruct'}


WARNING 07-28 05:47:47 [arg_utils.py:1631] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 07-28 05:47:48 [model.py:623] Resolved architecture: Qwen2_5_VLForConditionalGeneration
WARNING 07-28 05:47:48 [model.py:2123] Casting torch.bfloat16 to torch.float16.
INFO 07-28 05:47:48 [model.py:1788] Using max model len 4096
INFO 07-28 05:47:48 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=14160.
INFO 07-28 05:47:49 [vllm.py:1109] Asynchronous scheduling is enabled.
INFO 07-28 05:47:49 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


INFO 07-28 05:48:02 [core.py:116] Initializing a V1 LLM engine (v0.26.0) with config: model='Qwen/Qwen2.5-VL-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-VL-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=bitsandbytes, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=bitsandbytes, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_en

[W728 05:48:03.757072566 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


...

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 07-28 10:30:06 [jit_monitor.py:135] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1/1 [00:11<00:00, 11.23s/it, est. speed input: 71.68 toks/s, output: 4.10 toks/s]

Checkpoint: batch 155/155, 2465 rows saved

PART B -- QWEN2.5-VL-7B CORRECTOR -- CONFIG 3
Image included: True | Template context: False
Rows processed: 2465 | Errors: 0

Tesseract baseline CER (Section 11):     0.7215
Tesseract CER on this joined subset:      0.7215
Corrected CER (config 3):                0.5984
Corrected WER (config 3):                0.9646
CER improvement (Tesseract - corrected):  +0.1230  (improved)
Rows where correction improved CER:        1815 / 2465
Rows where correction made it worse:       476 / 2465
Rows unchanged:                             174 / 2465
Empty corrected predictions:                0
Exact match to ground truth:                0.00%

Saved: /kaggle/working/qwen25vl_partB_config3_predictions.csv
Saved: /kaggle/working/qwen25vl_partB_config3_summary.txt

This corrector is scoped to Config 3 only for this project -- see scope note
at the top of this file before rerunning with a different CONFIG_ID.


In [3]:
"""
Part B -- Config 3 (Qwen2.5-VL corrector, image + text, no template context)
Qualitative error analysis -- same style as Sections 10, 11, 16, 20, 26, 27's
manual worst/best inspection, applied here to the second Config-3 corrector.

IMPORTANT (per Section 27's methodological finding): the single-word repetition
detector used in the Config 1 analysis script under-detects multi-word phrase
loops, which is exactly the failure mode Config 3 (image-grounded) tends to
produce instead of free-form fabrication. This script uses the CORRECTED n-gram
detector from Section 27 (n=2-6, repeats >=3 times, covers >=40% of output
tokens), not the original single-word heuristic -- do not revert to the
Config 1 version's `len(set(words)) <= 2` check for this config.

Before trusting this detector's headline number on a new model, spot-check it
in both directions (does it catch known loops, does it avoid flagging normal
short repeated function words) the same way Section 27 did -- don't assume the
Qwen3-VL validation transfers automatically to a different model's outputs.
"""

import re
from collections import Counter
from pathlib import Path
import pandas as pd

PRED_CSV = Path("/kaggle/working/qwen25vl_partB_config3_predictions.csv")
OUT_TXT  = Path("/kaggle/working/qwen25vl_partB_config3_error_analysis.txt")

df = pd.read_csv(PRED_CSV)
df["ground_truth"] = df["ground_truth"].fillna("")
df["corrected_prediction"] = df["corrected_prediction"].fillna("")
df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")

# ============================================================
# 1. Exact-match diagnosis -- is 0% real, or a punctuation-stripping artifact?
# ============================================================
PUNCT_RE = re.compile(r"['\u2018\u2019\u061B\u060C\u06D4۔]")  # quotes, Urdu comma/semicolon/full-stop

def strip_punct(s):
    return PUNCT_RE.sub("", s).strip()

df["gt_stripped"] = df["ground_truth"].apply(strip_punct)
df["pred_stripped"] = df["corrected_prediction"].apply(strip_punct)
exact_raw = (df["corrected_prediction"] == df["ground_truth"]).sum()
exact_stripped = (df["pred_stripped"] == df["gt_stripped"]).sum()

# ============================================================
# 2. CORRECTED repetition-loop detector (Section 27) -- n-gram based, n=2..6,
# flags a prediction if any n-gram repeats >=3 times AND those repeats cover
# >=40% of the total tokens in the output. Replaces the Config 1 script's
# single-word `len(set(words)) <= 2` heuristic, which Section 27 found gives
# false negatives on multi-word phrase loops.
# ============================================================
def is_repetition_loop(text, min_repeats=3, coverage_thresh=0.40, ngram_range=(2, 6)):
    words = text.split()
    n_words = len(words)
    if n_words < min_repeats * 2:
        return False
    for n in range(ngram_range[0], min(ngram_range[1], n_words) + 1):
        ngrams = [tuple(words[i:i+n]) for i in range(n_words - n + 1)]
        counts = Counter(ngrams)
        for gram, cnt in counts.items():
            if cnt >= min_repeats:
                covered_tokens = cnt * n
                if covered_tokens / n_words >= coverage_thresh:
                    return True
    return False

df["is_repetition_loop"] = df["corrected_prediction"].apply(is_repetition_loop)

# --- Validation step, per Section 27's methodological note: run this BEFORE
# trusting the headline repetition-loop rate. Populate these lists manually
# from a worst-10 inspection pass (see section 3 below) before reporting a
# final number -- flagged here as a required manual step, not automated.
# known_loop_file_ids = [...]      # fill in after manual inspection
# known_nonloop_file_ids = [...]   # fill in after manual inspection
# then check df.set_index("file_id").loc[known_loop_file_ids, "is_repetition_loop"].all()
# and NOT df.set_index("file_id").loc[known_nonloop_file_ids, "is_repetition_loop"].any()

# ============================================================
# 3. Worst / best examples (matching Section 10/11/16/20/26/27's format)
# ============================================================
worst10 = df.nlargest(10, "cer_sensitive")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "cer_sensitive", "tesseract_cer_sensitive", "is_repetition_loop"]
]
best10_nonperfect = df[df["cer_sensitive"] > 0].nsmallest(10, "cer_sensitive")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "cer_sensitive", "tesseract_cer_sensitive"]
]

# ============================================================
# 4. Where correction helped most / hurt most (delta vs Tesseract)
# ============================================================
df["delta"] = df["tesseract_cer_sensitive"] - df["cer_sensitive"]  # positive = improved
biggest_gains = df.nlargest(10, "delta")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "tesseract_cer_sensitive", "cer_sensitive", "delta"]
]
biggest_regressions = df.nsmallest(10, "delta")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "tesseract_cer_sensitive", "cer_sensitive", "delta", "is_repetition_loop"]
]

# ============================================================
# 5. Failure-mode categorization -- uses the corrected repetition-loop flag
# ============================================================
def categorize(row):
    pred, gt, tess = row["corrected_prediction"], row["ground_truth"], row["tesseract_prediction"]
    if pred == "":
        return "empty_output"
    if pred == tess:
        return "unchanged_copy_of_tesseract"
    if pred.strip() == gt.strip():
        return "exact_after_strip"
    if row["is_repetition_loop"]:
        return "repetition_loop"
    if row["cer_sensitive"] >= 1.5:
        return "fabrication_worse_than_tesseract_non_loop"
    if row["delta"] > 0.1:
        return "meaningful_improvement"
    if row["delta"] < -0.1:
        return "meaningful_regression"
    return "marginal_change"

df["failure_category"] = df.apply(categorize, axis=1)
category_counts = df["failure_category"].value_counts()

# ============================================================
# 6. Per-writer breakdown (writers with >=5 samples), matching Section 10/11 style
# ============================================================
writer_stats = (
    df.groupby("writer_id")
    .agg(n=("cer_sensitive", "size"), mean_cer=("cer_sensitive", "mean"), mean_delta=("delta", "mean"))
    .query("n >= 5")
    .sort_values("mean_cer", ascending=False)
)

# ============================================================
# 7. Cross-model comparison line (Qwen3-VL Config 3, Section 27 -- for the
# paper's generalization claim: does the pattern hold on a weaker corrector?)
# ============================================================
QWEN3VL_CONFIG3_CER = 0.4829       # Section 27
QWEN3VL_CONFIG3_LOOP_PCT = 1.34    # Section 27, repetition-loop category %
this_model_cer = df["cer_sensitive"].mean()
this_model_loop_pct = 100 * df["is_repetition_loop"].mean()

# ============================================================
# WRITE REPORT
# ============================================================
lines = []
lines.append("="*70)
lines.append("PART B -- CONFIG 3 (QWEN2.5-VL, IMAGE+TEXT, NO TEMPLATE CONTEXT)")
lines.append("QUALITATIVE ERROR ANALYSIS")
lines.append("="*70)

lines.append("\n--- 1. Exact-match diagnosis ---")
lines.append(f"Exact match, raw strings:            {exact_raw} / {len(df)} ({100*exact_raw/len(df):.2f}%)")
lines.append(f"Exact match, punctuation-stripped:    {exact_stripped} / {len(df)} ({100*exact_stripped/len(df):.2f}%)")
if exact_stripped > exact_raw:
    lines.append("--> Some 'failures' are punctuation-only mismatches (quotes/full-stop), not content errors.")
else:
    lines.append("--> 0% exact match is NOT a punctuation artifact -- genuine content mismatches throughout.")

lines.append("\n--- 2. Failure-mode category counts (corrected repetition-loop detector) ---")
for cat, n in category_counts.items():
    lines.append(f"  {cat:38s}: {n:5d} ({100*n/len(df):.1f}%)")

lines.append("\n--- 2b. REQUIRED MANUAL VALIDATION STEP (Section 27 discipline) ---")
lines.append("Before reporting the repetition-loop % above, manually inspect the worst-10")
lines.append("predictions below and confirm: (a) every genuine loop is flagged True, and")
lines.append("(b) short legitimate corrections that reuse common Urdu function words are")
lines.append("flagged False. Do not trust this number unvalidated -- see Section 27, where")
lines.append("an unvalidated looser detector returned a misleading 65.7% 'repetition rate'")
lines.append("that was actually just ordinary grammar.")

lines.append("\n--- 3. Worst 10 predictions (by corrected CER) ---")
for _, r in worst10.iterrows():
    lines.append(f"\n  [{r['file_id']}] corrected_CER={r['cer_sensitive']:.3f} | tesseract_CER={r['tesseract_cer_sensitive']:.3f} | flagged_loop={r['is_repetition_loop']}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 4. Best 10 non-perfect predictions ---")
for _, r in best10_nonperfect.iterrows():
    lines.append(f"\n  [{r['file_id']}] corrected_CER={r['cer_sensitive']:.3f} | tesseract_CER={r['tesseract_cer_sensitive']:.3f}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 5. Biggest 10 improvements over Tesseract (delta = tess_CER - corrected_CER) ---")
for _, r in biggest_gains.iterrows():
    lines.append(f"\n  [{r['file_id']}] delta={r['delta']:+.3f}  (tess={r['tesseract_cer_sensitive']:.3f} -> corrected={r['cer_sensitive']:.3f})")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 6. Biggest 10 regressions vs Tesseract ---")
for _, r in biggest_regressions.iterrows():
    lines.append(f"\n  [{r['file_id']}] delta={r['delta']:+.3f}  (tess={r['tesseract_cer_sensitive']:.3f} -> corrected={r['cer_sensitive']:.3f}) | flagged_loop={r['is_repetition_loop']}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 7. Per-writer CER (writers with >=5 samples, worst 10) ---")
lines.append(writer_stats.head(10).to_string())

lines.append("\n--- 8. Cross-model comparison (generalization check vs. Qwen3-VL Config 3, Section 27) ---")
lines.append(f"  Qwen3-VL-8B   Config 3 CER:        {QWEN3VL_CONFIG3_CER:.4f}")
lines.append(f"  Qwen2.5-VL-7B Config 3 CER:        {this_model_cer:.4f}")
lines.append(f"  Qwen3-VL-8B   repetition-loop %:   {QWEN3VL_CONFIG3_LOOP_PCT:.2f}%")
lines.append(f"  Qwen2.5-VL-7B repetition-loop %:   {this_model_loop_pct:.2f}%")
lines.append("  (Interpretation left for manual write-up: does the weaker corrector show the")
lines.append("   same 'visual grounding helps, but a residual repetition-loop tail persists'")
lines.append("   pattern as Qwen3-VL, or a different failure shape?)")

report = "\n".join(lines)
print(report)
OUT_TXT.write_text(report, encoding="utf-8")
print(f"\n\nSaved full report to: {OUT_TXT}")

PART B -- CONFIG 3 (QWEN2.5-VL, IMAGE+TEXT, NO TEMPLATE CONTEXT)
QUALITATIVE ERROR ANALYSIS

--- 1. Exact-match diagnosis ---
Exact match, raw strings:            0 / 2465 (0.00%)
Exact match, punctuation-stripped:    0 / 2465 (0.00%)
--> 0% exact match is NOT a punctuation artifact -- genuine content mismatches throughout.

--- 2. Failure-mode category counts (corrected repetition-loop detector) ---
  meaningful_improvement                :  1386 (56.2%)
  marginal_change                       :   740 (30.0%)
  meaningful_regression                 :   173 (7.0%)
  unchanged_copy_of_tesseract           :    81 (3.3%)
  repetition_loop                       :    77 (3.1%)
  fabrication_worse_than_tesseract_non_loop:     8 (0.3%)

--- 2b. REQUIRED MANUAL VALIDATION STEP (Section 27 discipline) ---
Before reporting the repetition-loop % above, manually inspect the worst-10
predictions below and confirm: (a) every genuine loop is flagged True, and
(b) short legitimate corrections that reu

# GEMINI

In [1]:
"""
Part B -- Zero-Shot Correction (RQ2)
Corrector: Gemini 3.5 Flash-Lite
Target: Tesseract's weakest-baseline predictions (Section 11, CER 0.7215)
Config: 3 (image + text, no template context) -- the last remaining Part B run
        for this paper's scope (Qwen3-VL Configs 1+3, Qwen2.5-VL Config 3, this run).

Structure/pipeline reused from the Qwen2.5-VL Part B script (Config 3): same
CER/WER helpers, same evaluation-set/Tesseract join logic, same output schema,
same base correction instruction, same validated repetition-loop detector and
category breakdown from Section 27.

NEW here, built on top of Section 23's multi-key approach:
- Any number of API keys can be supplied; the not-yet-done rows are split
  dynamically into that many roughly-equal shards (e.g. 6 keys -> ~411 rows
  each for a 2,465-row set).
- Each key is pre-flight validated BEFORE any rows are assigned to it --
  Section 23's first retry attempt wasted a full 320-row pass on a revoked
  key; this catches that class of failure before it costs anything.
- Unlike Section 23 (keys processed sequentially in one notebook pass),
  shards here run in parallel threads, one per key. These are network-bound
  API calls, not GPU-bound work, so parallelizing is safe and should cut
  total wall time by roughly Nx versus running keys one after another.
- Each key respects its own 15 RPM limit (Section 23's confirmed free-tier
  cap) via a per-thread sliding-window rate limiter.
- Resume-safe: rerunning only processes file_ids that don't already have a
  successful (non-error) row somewhere across the shard files / merged
  output -- so you can top up quota, swap a dead key, and rerun freely.
"""

# pip install -q -U google-genai   # if not already installed in this environment

import os, re, time, threading
from pathlib import Path
import pandas as pd
from PIL import Image

from google import genai
from google.genai import types

# ============================================================
# CONFIG -- paste in as many keys as you have. Any N works; images are
# divided dynamically across however many keys you provide below.
# ============================================================
import os

def _load_gemini_keys():
    """Read comma-separated Gemini API keys from the environment, or from
    Kaggle Secrets (Add-ons -> Secrets, label: GEMINI_API_KEYS). Never hardcode keys."""
    raw = os.environ.get("GEMINI_API_KEYS", "")
    if not raw:
        try:
            from kaggle_secrets import UserSecretsClient
            raw = UserSecretsClient().get_secret("GEMINI_API_KEYS")
        except Exception:
            raw = ""
    return [k.strip() for k in raw.split(",") if k.strip()]

GEMINI_API_KEYS = _load_gemini_keys()
if not GEMINI_API_KEYS:
    raise RuntimeError("No Gemini API keys found. Set GEMINI_API_KEYS (comma-separated) "
                       "as an environment variable or Kaggle Secret.")

MODEL_ID = "gemini-3.5-flash-lite"
CONFIG_ID = 3  # image + text, no template context -- fixed for this run

DATASET_ROOT = Path("/kaggle/input/datasets/hy7haseeb/handwritten-dataset/kaggle_upload")
IMAGES_DIR   = DATASET_ROOT / "images"
EVAL_CSV     = DATASET_ROOT / "evaluation_set.csv"

TESSERACT_ROOT = Path("/kaggle/input/datasets/hy7haseeb/tesseract-real-prediction")
TESSERACT_CSV  = TESSERACT_ROOT / "tesseract_predictions.csv"

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)
FINAL_PRED_CSV = OUT_DIR / f"gemini_partB_config{CONFIG_ID}_predictions.csv"
SUMMARY_TXT    = OUT_DIR / f"gemini_partB_config{CONFIG_ID}_summary.txt"
SHARD_STEM     = f"gemini_partB_config{CONFIG_ID}_shard"  # + _{i}.csv

RPM_PER_KEY = 15          # Section 23's confirmed free-tier limit
RPD_PER_KEY = 500         # daily cap, informational -- flagged if a shard would exceed it
CHECKPOINT_EVERY = 25     # same cadence as Section 23
MAX_OUTPUT_TOKENS = 128
TESSERACT_BASELINE_CER = 0.7215

# ============================================================
# CER / WER HELPERS -- identical to every other script in this project
# ============================================================
DIACRITIC_RE = re.compile(r"[\u064B-\u065F\u0670]")

def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    if len(gt) == 0: return 1.0 if len(pred) > 0 else 0.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    p, g = pred.split(), gt.split()
    if len(g) == 0: return 1.0 if len(p) > 0 else 0.0
    return edit_distance(p, g) / len(g)

# ============================================================
# LOAD + JOIN (identical logic to the Qwen Part B scripts)
# ============================================================
eval_df = pd.read_csv(EVAL_CSV)
tess_df = pd.read_csv(TESSERACT_CSV)[["file_id", "prediction"]].rename(
    columns={"prediction": "tesseract_prediction"}
)
df = eval_df.merge(tess_df, on="file_id", how="inner")
print(f"Eval set rows: {len(eval_df)} | Tesseract rows: {len(tess_df)} | Joined: {len(df)}")
if len(df) != len(eval_df):
    print(f"WARNING: {len(eval_df) - len(df)} rows dropped in the join -- check file_id alignment.")
df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")

disk_files = {p.name: p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}
df["resolved_path"] = df["image_path"].apply(lambda p: disk_files.get(str(p).replace("\\", "/").split("/")[-1]))
missing = df["resolved_path"].isna().sum()
print(f"Missing images: {missing}")
df = df.dropna(subset=["resolved_path"]).reset_index(drop=True)

# ============================================================
# PROMPT -- same base correction instruction as the Qwen Part B scripts,
# Config 3 (image + text, no template-sentence context). Delivered via
# system_instruction, matching how Gemini received its Part A prompt
# (Section 23) and how each model's delivery mechanism was kept consistent
# with what its API actually supports (Section 12).
# ============================================================
BASE_INSTRUCTION = (
    "You are correcting the output of a low-quality OCR system that transcribed a single "
    "line of handwritten Urdu text in the Nastaliq script. The OCR system's guess is often "
    "wrong -- garbled, incomplete, or entirely incorrect. Your job is to produce the correct "
    "transcription. Output ONLY the corrected Urdu text -- no explanation, no commentary."
)

# ============================================================
# FIND ALREADY-DONE ROWS (merged across any prior shard files / final file)
# ============================================================
def load_existing():
    frames = []
    if FINAL_PRED_CSV.exists():
        frames.append(pd.read_csv(FINAL_PRED_CSV))
    for shard_path in OUT_DIR.glob(f"{SHARD_STEM}_*.csv"):
        frames.append(pd.read_csv(shard_path))
    if not frames:
        return pd.DataFrame(columns=["file_id", "error"])
    combined = pd.concat(frames, ignore_index=True)
    combined["error"] = combined["error"].fillna("")
    # only rows with no error count as "done" -- an errored row should be retried
    combined = combined[combined["error"] == ""]
    return combined.drop_duplicates(subset=["file_id"])

existing = load_existing()
done_ids = set(existing["file_id"].astype(str))
print(f"Already completed (no error): {len(done_ids)} / {len(df)}")

remaining = df[~df["file_id"].astype(str).isin(done_ids)].reset_index(drop=True)
print(f"Remaining to process: {len(remaining)}")

# ============================================================
# PRE-FLIGHT KEY VALIDATION (Section 23's lesson: a bad key wastes a whole
# chunk if not caught before committing rows to it)
# ============================================================
def validate_key(key, idx):
    try:
        client = genai.Client(api_key=key)
        client.models.generate_content(
            model=MODEL_ID,
            contents=["ping"],
            config=types.GenerateContentConfig(max_output_tokens=5),
        )
        return True
    except Exception as e:
        print(f"  Key {idx+1}: INVALID -- {str(e)[:150]}")
        return False

print(f"\nValidating {len(GEMINI_API_KEYS)} API key(s) before assigning any work...")
valid_keys = []
for i, k in enumerate(GEMINI_API_KEYS):
    if validate_key(k, i):
        print(f"  Key {i+1}: OK")
        valid_keys.append(k)

if not valid_keys:
    raise RuntimeError("No valid API keys -- aborting before any images were processed.")

N = len(valid_keys)
print(f"\n{N} valid key(s) of {len(GEMINI_API_KEYS)} provided. Dividing {len(remaining)} "
      f"remaining images into {N} shard(s).")

if N and (-(-len(remaining) // N)) > RPD_PER_KEY:
    print(f"WARNING: each key's share (~{-(-len(remaining)//N)}) exceeds the {RPD_PER_KEY}/day "
          f"free-tier cap (Section 23). Add more keys, or expect some 429s partway through a "
          f"shard -- the script checkpoints as it goes, so a rerun with topped-up keys will pick "
          f"up exactly where it left off.")

# ============================================================
# BUILD SHARDS -- interleaved split keeps shard sizes balanced (differ by
# at most 1 row), and is dynamic for any N.
# ============================================================
shards = []
for i in range(N):
    shard_df = remaining.iloc[i::N].reset_index(drop=True)
    shards.append(shard_df)
    print(f"  Shard {i+1} (key {i+1}): {len(shard_df)} images")

# ============================================================
# RATE LIMITER -- sliding-window, one instance per key/thread
# ============================================================
class RateLimiter:
    def __init__(self, max_per_minute):
        self.max_per_minute = max_per_minute
        self.timestamps = []
        self.lock = threading.Lock()

    def wait(self):
        sleep_for = 0
        with self.lock:
            now = time.time()
            self.timestamps = [t for t in self.timestamps if now - t < 60]
            if len(self.timestamps) >= self.max_per_minute:
                sleep_for = 60 - (now - self.timestamps[0]) + 0.1
            self.timestamps.append(now + sleep_for)
        if sleep_for > 0:
            time.sleep(sleep_for)

# ============================================================
# PER-SHARD WORKER
# ============================================================
print_lock = threading.Lock()

def process_shard(shard_idx, shard_df, api_key):
    shard_csv = OUT_DIR / f"{SHARD_STEM}_{shard_idx}.csv"
    results = []
    if shard_csv.exists():
        prior = pd.read_csv(shard_csv)
        prior["error"] = prior["error"].fillna("")
        results = prior.to_dict("records")
        done_here = {str(r["file_id"]) for r in results if not r["error"]}
        shard_df = shard_df[~shard_df["file_id"].astype(str).isin(done_here)].reset_index(drop=True)

    if len(shard_df) == 0:
        with print_lock:
            print(f"[Shard {shard_idx+1}] nothing to do")
        return

    client = genai.Client(api_key=api_key)
    limiter = RateLimiter(RPM_PER_KEY)

    for n, row in enumerate(shard_df.itertuples(), 1):
        limiter.wait()
        gt = str(row.text)
        try:
            img = Image.open(row.resolved_path).convert("RGB")
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=[f"OCR guess: {row.tesseract_prediction}", img],
                config=types.GenerateContentConfig(
                    system_instruction=BASE_INSTRUCTION,
                    temperature=0.0,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                ),
            )
            pred_text = (response.text or "").strip()
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": pred_text,
                "cer_sensitive": cer(pred_text, gt, False),
                "cer_insensitive": cer(pred_text, gt, True),
                "wer_sensitive": wer(pred_text, gt, False),
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "key_used": shard_idx + 1,
                "error": "",
            })
        except Exception as e:
            results.append({
                "sentence_id": row.sentence_id,
                "file_id": row.file_id,
                "writer_id": row.writer_id,
                "ground_truth": gt,
                "tesseract_prediction": row.tesseract_prediction,
                "corrected_prediction": "",
                "cer_sensitive": 1.0,
                "cer_insensitive": 1.0,
                "wer_sensitive": 1.0,
                "tesseract_cer_sensitive": cer(row.tesseract_prediction, gt, False),
                "config_id": CONFIG_ID,
                "key_used": shard_idx + 1,
                "error": str(e)[:300],
            })

        if n % CHECKPOINT_EVERY == 0 or n == len(shard_df):
            pd.DataFrame(results).to_csv(shard_csv, index=False)
            with print_lock:
                print(f"[Shard {shard_idx+1}] {n}/{len(shard_df)} done, {len(results)} rows saved")

    pd.DataFrame(results).to_csv(shard_csv, index=False)
    with print_lock:
        print(f"[Shard {shard_idx+1}] COMPLETE -- {len(results)} rows")

# ============================================================
# RUN SHARDS IN PARALLEL -- one thread per key, independent quotas
# ============================================================
if len(remaining) > 0:
    threads = [threading.Thread(target=process_shard, args=(i, shards[i], valid_keys[i]))
               for i in range(N)]
    for t in threads: t.start()
    for t in threads: t.join()
else:
    print("Nothing left to do -- skipping straight to merge + summary.")

# ============================================================
# MERGE ALL SHARDS -> FINAL CSV (success rows win over error rows for the
# same file_id; dedupe after that)
# ============================================================
shard_frames = [pd.read_csv(p) for p in OUT_DIR.glob(f"{SHARD_STEM}_*.csv")]
if FINAL_PRED_CSV.exists():
    shard_frames.append(pd.read_csv(FINAL_PRED_CSV))
raw_all = pd.concat(shard_frames, ignore_index=True) if shard_frames else pd.DataFrame()
raw_all["error"] = raw_all["error"].fillna("")
raw_all["_is_error"] = raw_all["error"] != ""
raw_all = raw_all.sort_values("_is_error").drop_duplicates(subset=["file_id"], keep="first")
raw_all = raw_all.drop(columns=["_is_error"]).reset_index(drop=True)
raw_all.to_csv(FINAL_PRED_CSV, index=False)

print(f"\nMerged final file: {FINAL_PRED_CSV} -- {len(raw_all)} rows "
      f"({len(df) - len(raw_all)} not yet covered)")
still_missing = set(df["file_id"].astype(str)) - set(raw_all["file_id"].astype(str))
err_remaining = (raw_all["error"] != "").sum()
if still_missing:
    print(f"WARNING: {len(still_missing)} file_ids still missing from the merged output -- "
          f"rerun the script (fresh/topped-up keys if these are quota failures) to fill them in.")
if err_remaining:
    print(f"WARNING: {err_remaining} rows still have an error after merge -- rerun to retry them.")

# ============================================================
# REPETITION-LOOP DETECTOR -- Section 27's VALIDATED version (n-gram,
# n=2-6, repeats >=3x, covers >=40% of output tokens). Deliberately NOT
# the naive single-word-uniqueness heuristic that gave a false negative
# in Section 27 before being caught and corrected.
# ============================================================
def is_repetition_loop(text, min_n=2, max_n=6, min_repeats=3, min_coverage=0.40):
    tokens = str(text).split()
    if len(tokens) < min_n * min_repeats:
        return False
    for n in range(min_n, max_n + 1):
        counts = {}
        for i in range(len(tokens) - n + 1):
            ng = tuple(tokens[i:i+n])
            counts[ng] = counts.get(ng, 0) + 1
        for ng, count in counts.items():
            if count >= min_repeats and (count * n) / len(tokens) >= min_coverage:
                return True
    return False

# ============================================================
# QUALITATIVE CATEGORY BREAKDOWN -- same categories as Sections 26-28
# ============================================================
def categorize(row):
    if row["error"]:
        return "error"
    pred = str(row["corrected_prediction"])
    if pred.strip() == str(row["tesseract_prediction"]).strip():
        return "unchanged_copy"
    if is_repetition_loop(pred):
        return "repetition_loop"
    if row["cer_sensitive"] >= 1.5:
        return "fabrication_nonloop"
    delta = row["tesseract_cer_sensitive"] - row["cer_sensitive"]
    if delta > 0.05:
        return "meaningful_improvement"
    if delta < -0.05:
        return "meaningful_regression"
    return "marginal_change"

final = pd.read_csv(FINAL_PRED_CSV)
final["error"] = final["error"].fillna("")
final["category"] = final.apply(categorize, axis=1)
ok = final[final["error"] == ""].copy()
ok["corrected_prediction"] = ok["corrected_prediction"].fillna("").astype(str)
ok["ground_truth"] = ok["ground_truth"].astype(str)

# ============================================================
# ENHANCED SUMMARY REPORT -- matches the format of Sections 26-28, plus
# per-key diagnostics (to catch shared-quota-pool issues like Section 23's
# keys 1 & 2)
# ============================================================
lines = []
lines.append("="*70)
lines.append(f"PART B -- GEMINI 3.5 FLASH-LITE CORRECTOR -- CONFIG {CONFIG_ID}")
lines.append("Image + text, no template-sentence context")
lines.append("="*70)
lines.append(f"Total rows: {len(final)} | Errors remaining: {(final['error']!='').sum()} | "
              f"Successful: {len(ok)}")
lines.append(f"Keys used: {N} valid of {len(GEMINI_API_KEYS)} provided")
lines.append("")
lines.append("Per-key diagnostics:")
for i in range(N):
    shard_csv = OUT_DIR / f"{SHARD_STEM}_{i}.csv"
    if shard_csv.exists():
        sdf = pd.read_csv(shard_csv)
        sdf["error"] = sdf["error"].fillna("")
        n_err = (sdf["error"] != "").sum()
        lines.append(f"  Key {i+1}: {len(sdf)} rows, {n_err} errors")
lines.append("")

if len(ok) > 0:
    lines.append(f"Tesseract baseline CER (Section 11):        {TESSERACT_BASELINE_CER:.4f}")
    lines.append(f"Tesseract CER on this joined subset:         {ok['tesseract_cer_sensitive'].mean():.4f}")
    lines.append(f"Corrected CER (Config {CONFIG_ID}):                    {ok['cer_sensitive'].mean():.4f}")
    lines.append(f"Corrected CER (diacritic-insensitive):       {ok['cer_insensitive'].mean():.4f}")
    lines.append(f"Corrected WER:                                {ok['wer_sensitive'].mean():.4f}")
    rel_reduction = 100 * (ok['tesseract_cer_sensitive'].mean() - ok['cer_sensitive'].mean()) / ok['tesseract_cer_sensitive'].mean()
    lines.append(f"Relative CER reduction vs. Tesseract:         {rel_reduction:+.1f}%")
    lines.append("")
    n_improved = (ok['cer_sensitive'] < ok['tesseract_cer_sensitive']).sum()
    n_worse = (ok['cer_sensitive'] > ok['tesseract_cer_sensitive']).sum()
    n_same = (ok['cer_sensitive'] == ok['tesseract_cer_sensitive']).sum()
    lines.append(f"Rows improved:   {n_improved} / {len(ok)} ({100*n_improved/len(ok):.1f}%)")
    lines.append(f"Rows worse:      {n_worse} / {len(ok)} ({100*n_worse/len(ok):.1f}%)")
    lines.append(f"Rows unchanged:  {n_same} / {len(ok)}")
    lines.append(f"Exact match:     {100*(ok['corrected_prediction'].str.strip()==ok['ground_truth'].str.strip()).mean():.2f}%")
    lines.append(f"Empty predictions: {(ok['corrected_prediction'].str.strip()=='').sum()}")
    lines.append("")
    lines.append("Category breakdown:")
    cat_counts = ok["category"].value_counts()
    for cat, count in cat_counts.items():
        lines.append(f"  {cat:28s} {count:5d}  ({100*count/len(ok):.2f}%)")
    lines.append("")
    lines.append("Comparison across Part B Config 3 runs (this paper's full scope):")
    lines.append(f"  {'Corrector':<24}{'CER':>10}{'Rel. reduction':>18}{'% improved':>14}")
    lines.append(f"  {'Qwen3-VL-8B':<24}{'0.4829':>10}{'~33%':>18}{'86.2%':>14}")
    lines.append(f"  {'Qwen2.5-VL-7B':<24}{'0.5984':>10}{'~17%':>18}{'73.6%':>14}")
    lines.append(f"  {'Gemini 3.5 Flash-Lite':<24}{ok['cer_sensitive'].mean():>10.4f}"
                  f"{f'{rel_reduction:.1f}%':>18}{100*n_improved/len(ok):>13.1f}%")
else:
    lines.append("No successful rows yet -- rerun with valid/topped-up keys.")

summary_text = "\n".join(lines)
print("\n" + summary_text)
SUMMARY_TXT.write_text(summary_text, encoding="utf-8")

final.to_csv(FINAL_PRED_CSV, index=False)  # persist the `category` column too

print(f"\nSaved: {FINAL_PRED_CSV}")
print(f"Saved: {SUMMARY_TXT}")
if still_missing or err_remaining:
    print("\nNOTE: some rows are still missing/erroring -- rerun this script (same keys or "
          "fresh ones) to fill them in; already-completed rows are skipped automatically.")
else:
    print("\nAll rows complete with 0 errors. Part B is now finished for this paper's scope "
          "(Qwen3-VL Configs 1+3, Qwen2.5-VL Config 3, Gemini Config 3).")

Eval set rows: 2465 | Tesseract rows: 2465 | Joined: 2465
Missing images: 0
Already completed (no error): 0 / 2465
Remaining to process: 2465

Validating 7 API key(s) before assigning any work...
  Key 1: OK
  Key 2: OK
  Key 3: OK
  Key 4: OK
  Key 5: OK
  Key 6: OK
  Key 7: OK

7 valid key(s) of 7 provided. Dividing 2465 remaining images into 7 shard(s).
  Shard 1 (key 1): 353 images
  Shard 2 (key 2): 352 images
  Shard 3 (key 3): 352 images
  Shard 4 (key 4): 352 images
  Shard 5 (key 5): 352 images
  Shard 6 (key 6): 352 images
  Shard 7 (key 7): 352 images
[Shard 5] 25/352 done, 25 rows saved
[Shard 1] 25/353 done, 25 rows saved
[Shard 4] 25/352 done, 25 rows saved
[Shard 2] 25/352 done, 25 rows saved
[Shard 7] 25/352 done, 25 rows saved
[Shard 6] 25/352 done, 25 rows saved
[Shard 3] 25/352 done, 25 rows saved
[Shard 6] 50/352 done, 50 rows saved
[Shard 1] 50/353 done, 50 rows saved
[Shard 4] 50/352 done, 50 rows saved
[Shard 5] 50/352 done, 50 rows saved
[Shard 7] 50/352 done, 5

In [ ]:
"""
Part B -- Gemini 3.5 Flash-Lite, Config 3
Retry script for rows that still have an error after the main run.

UPDATED: reads the predictions CSV from the recovered input dataset
(/kaggle/input/datasets/hy7haseeb/real-part-b/), since /kaggle/input is
read-only -- copies it into /kaggle/working first and works from that copy
from then on. If the file already shows 0 errors (it may, if the recovered
snapshot is from after your last successful retry pass), this script will
just report that and exit cleanly -- nothing will be re-sent to the API.

What it does:
1. Loads the predictions CSV and isolates only the error rows, if any.
2. Prints the actual error messages, grouped/counted.
3. Retries those rows only, round-robin across your valid keys, respecting
   each key's 15 RPM limit.
4. Writes the fixes back into the working copy and reprints the updated
   aggregate CER.

IMPORTANT: once this finishes with 0 errors, hit Save Version on the
notebook right away -- that's what actually persists /kaggle/working past
a session recycle, not just having the file sit in the live panel.
"""

import re, time, shutil
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image

from google import genai
from google.genai import types

# ============================================================
# CONFIG
# ============================================================


MODEL_ID = "gemini-3.5-flash-lite"
CONFIG_ID = 3

DATASET_ROOT = Path("/kaggle/input/datasets/hy7haseeb/handwritten-dataset/kaggle_upload")
IMAGES_DIR   = DATASET_ROOT / "images"
EVAL_CSV     = DATASET_ROOT / "evaluation_set.csv"
TESSERACT_ROOT = Path("/kaggle/input/datasets/hy7haseeb/tesseract-real-prediction")
TESSERACT_CSV  = TESSERACT_ROOT / "tesseract_predictions.csv"

# NEW -- recovered predictions file lives in the read-only input dataset;
# copy it into /kaggle/working (writable) before doing anything else.
SOURCE_CSV = Path("/kaggle/input/datasets/hy7haseeb/real-part-b/gemini_partB_config3_predictions.csv")
OUT_DIR = Path("/kaggle/working")
FINAL_PRED_CSV = OUT_DIR / SOURCE_CSV.name

if not FINAL_PRED_CSV.exists():
    shutil.copy(SOURCE_CSV, FINAL_PRED_CSV)
    print(f"Copied {SOURCE_CSV.name} into /kaggle/working -- working from this writable copy.")
else:
    print(f"{FINAL_PRED_CSV.name} already present in /kaggle/working -- using existing copy "
          f"(delete it first if you want a fresh copy from the input dataset).")

RPM_PER_KEY = 15
MAX_OUTPUT_TOKENS = 128

BASE_INSTRUCTION = (
    "You are correcting the output of a low-quality OCR system that transcribed a single "
    "line of handwritten Urdu text in the Nastaliq script. The OCR system's guess is often "
    "wrong -- garbled, incomplete, or entirely incorrect. Your job is to produce the correct "
    "transcription. Output ONLY the corrected Urdu text -- no explanation, no commentary."
)

DIACRITIC_RE = re.compile(r"[\u064B-\u065F\u0670]")

def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    if len(gt) == 0: return 1.0 if len(pred) > 0 else 0.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt, strip=False):
    if strip: pred, gt = DIACRITIC_RE.sub("", pred), DIACRITIC_RE.sub("", gt)
    p, g = pred.split(), gt.split()
    if len(g) == 0: return 1.0 if len(p) > 0 else 0.0
    return edit_distance(p, g) / len(g)

# ============================================================
# LOAD CSV, ISOLATE ERROR ROWS
# ============================================================
final = pd.read_csv(FINAL_PRED_CSV)
final["error"] = final["error"].fillna("")
err_rows = final[final["error"] != ""].copy()

print(f"Total rows: {len(final)} | Currently erroring: {len(err_rows)}")

if len(err_rows) == 0:
    print("No error rows -- nothing to retry. Run the error-analysis script next.")
else:
    print("\nError message breakdown (grouped by first 60 chars):")
    for msg, count in Counter(err_rows["error"].str[:60]).most_common():
        print(f"  {count:3d}x  {msg}")

    eval_df = pd.read_csv(EVAL_CSV)
    tess_df = pd.read_csv(TESSERACT_CSV)[["file_id", "prediction"]].rename(
        columns={"prediction": "tesseract_prediction"})
    ref = eval_df.merge(tess_df, on="file_id", how="inner")
    disk_files = {p.name: p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}
    ref["resolved_path"] = ref["image_path"].apply(
        lambda p: disk_files.get(str(p).replace("\\", "/").split("/")[-1]))

    retry_df = err_rows[["file_id"]].merge(ref, on="file_id", how="left").reset_index(drop=True)
    missing_img = retry_df["resolved_path"].isna().sum()
    if missing_img:
        print(f"WARNING: {missing_img} error row(s) have no resolvable image -- skipping "
              f"those, check IMAGES_DIR / dataset mount.")
    retry_df = retry_df.dropna(subset=["resolved_path"]).reset_index(drop=True)

    # ============================================================
    # VALIDATE KEYS, THEN RETRY ROUND-ROBIN ACROSS THEM
    # ============================================================
    def validate_key(key, idx):
        try:
            client = genai.Client(api_key=key)
            client.models.generate_content(
                model=MODEL_ID, contents=["ping"],
                config=types.GenerateContentConfig(max_output_tokens=5))
            return True
        except Exception as e:
            print(f"  Key {idx+1}: INVALID -- {str(e)[:150]}")
            return False

    print(f"\nValidating {len(GEMINI_API_KEYS)} key(s)...")
    valid_keys = [k for i, k in enumerate(GEMINI_API_KEYS) if validate_key(k, i)]
    if not valid_keys:
        raise RuntimeError("No valid keys -- can't retry.")
    print(f"{len(valid_keys)} valid key(s). Retrying {len(retry_df)} row(s), round-robin, "
          f"{RPM_PER_KEY} RPM per key.")

    clients = [genai.Client(api_key=k) for k in valid_keys]
    last_call = [0.0] * len(clients)
    min_gap = 60.0 / RPM_PER_KEY

    retried = {}
    for n, row in enumerate(retry_df.itertuples()):
        ki = n % len(clients)
        gap = time.time() - last_call[ki]
        if gap < min_gap:
            time.sleep(min_gap - gap)
        gt = str(row.text)
        try:
            img = Image.open(row.resolved_path).convert("RGB")
            response = clients[ki].models.generate_content(
                model=MODEL_ID,
                contents=[f"OCR guess: {row.tesseract_prediction}", img],
                config=types.GenerateContentConfig(
                    system_instruction=BASE_INSTRUCTION,
                    temperature=0.0,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                ),
            )
            pred_text = (response.text or "").strip()
            row_cer = cer(pred_text, gt, False)
            retried[str(row.file_id)] = {
                "corrected_prediction": pred_text,
                "cer_sensitive": row_cer,
                "cer_insensitive": cer(pred_text, gt, True),
                "wer_sensitive": wer(pred_text, gt, False),
                "error": "",
            }
            print(f"  [{n+1}/{len(retry_df)}] {row.file_id}: OK (CER {row_cer:.3f})")
        except Exception as e:
            retried[str(row.file_id)] = {"error": str(e)[:300]}
            print(f"  [{n+1}/{len(retry_df)}] {row.file_id}: still failing -- {str(e)[:100]}")
        last_call[ki] = time.time()

    # ============================================================
    # APPLY RESULTS BACK INTO THE WORKING CSV
    # ============================================================
    final = final.set_index(final["file_id"].astype(str))
    n_fixed = 0
    for fid, upd in retried.items():
        if upd.get("error", "") == "":
            for col, val in upd.items():
                final.loc[fid, col] = val
            n_fixed += 1
        else:
            final.loc[fid, "error"] = upd["error"]
    final = final.reset_index(drop=True)
    final.to_csv(FINAL_PRED_CSV, index=False)

    still_err = (final["error"].fillna("") != "").sum()
    print(f"\nRetry complete: {n_fixed}/{len(retry_df)} row(s) fixed. "
          f"{still_err} row(s) still erroring after this pass.")

    ok = final[final["error"].fillna("") == ""]
    print(f"\nUpdated aggregate (successful rows only, n={len(ok)}):")
    print(f"  Corrected CER: {ok['cer_sensitive'].mean():.4f}")
    if "tesseract_cer_sensitive" in ok.columns:
        print(f"  Tesseract CER (same subset): {ok['tesseract_cer_sensitive'].mean():.4f}")

    if still_err:
        print("\nRerun this script again to retry the remainder.")
    else:
        print("\nAll rows now successful. SAVE VERSION NOW, then run the error-analysis script.")

Copied gemini_partB_config3_predictions.csv into /kaggle/working -- working from this writable copy.
Total rows: 2465 | Currently erroring: 32

Error message breakdown (grouped by first 60 chars):
   32x  429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '

Validating 7 key(s)...
7 valid key(s). Retrying 32 row(s), round-robin, 15 RPM per key.
  [1/32] 406_6: OK (CER 0.053)
  [2/32] 279_8: OK (CER 0.358)
  [3/32] 402_6: OK (CER 0.056)
  [4/32] 041_004_5: OK (CER 0.108)
  [5/32] 305_14: OK (CER 0.833)
  [6/32] 306_16: OK (CER 0.085)
  [7/32] 253_19: OK (CER 0.056)
  [8/32] 255_5: OK (CER 0.132)
  [9/32] 288_4: OK (CER 0.016)
  [10/32] 409_6: OK (CER 0.000)
  [11/32] 373_12: OK (CER 0.214)
  [12/32] 370_9: OK (CER 0.180)
  [13/32] 257_11: OK (CER 0.273)
  [14/32] 305_15: OK (CER 0.775)
  [15/32] 421_6: OK (CER 0.106)
  [16/32] 392_13: OK (CER 0.188)
  [17/32] 226_2: OK (CER 0.312)
  [18/32] 469_12: OK (CER 0.000)
  [19/32] 354_16: OK (CER 0.172)
  [20/32] 365_11: OK (CER 0.283)


In [2]:
"""
Part B -- Config 3 (Gemini 3.5 Flash-Lite corrector, image + text, no template context)
Qualitative error analysis -- same style as Sections 10, 11, 16, 20, 26, 27, 28's
manual worst/best inspection, applied here to the third and final Config-3 corrector.

UPDATED: reads the predictions CSV from the recovered input dataset
(/kaggle/input/datasets/hy7haseeb/real-part-b/), since /kaggle/input is
read-only -- copies it into /kaggle/working first, same pattern as the retry
script. Run the retry script first if that file still shows any error rows.

Mirrors the Qwen2.5-VL Config 3 analysis script structurally (same categorize()
logic, same corrected n-gram repetition-loop detector from Section 27, same
worst/best/delta/per-writer sections). Two adaptations for Gemini specifically:

1. Error rows (API failures) are excluded from the analysis below and counted
   separately, rather than assumed to be zero as the Qwen scripts could assume.
2. The cross-model comparison (Section 8 below) is three-way -- Gemini is the
   last of the three Config 3 correctors, so this is the first analysis script
   that can show the full weak-to-strong curve in one place.

IMPORTANT (per Section 27's own discipline): the repetition-loop detector's
validation on Qwen3-VL/Qwen2.5-VL's outputs does NOT automatically transfer to
Gemini's. Gemini's overall catastrophic-failure rate is much lower, so there
may be little to validate against here -- if the worst-10 list below contains
no genuine loops at all, that's expected given how clean this corrector's
output is, not a sign the detector is broken. Spot-check it anyway.
"""

import re, shutil
from collections import Counter
from pathlib import Path
import pandas as pd

# NEW -- recovered predictions file lives in the read-only input dataset;
# copy it into /kaggle/working (writable) before doing anything else.
SOURCE_CSV = Path("/kaggle/input/datasets/hy7haseeb/real-part-b/gemini_partB_config3_predictions.csv")
PRED_CSV = Path("/kaggle/working") / SOURCE_CSV.name
OUT_TXT  = Path("/kaggle/working/gemini_partB_config3_error_analysis.txt")

if not PRED_CSV.exists():
    shutil.copy(SOURCE_CSV, PRED_CSV)
    print(f"Copied {SOURCE_CSV.name} into /kaggle/working -- working from this writable copy.")
else:
    print(f"{PRED_CSV.name} already present in /kaggle/working -- using existing copy.")

df = pd.read_csv(PRED_CSV)
df["ground_truth"] = df["ground_truth"].fillna("")
df["corrected_prediction"] = df["corrected_prediction"].fillna("")
df["tesseract_prediction"] = df["tesseract_prediction"].fillna("")
df["error"] = df["error"].fillna("")

# ============================================================
# 0. Exclude error rows from the qualitative analysis
# ============================================================
n_total = len(df)
err_df = df[df["error"] != ""]
df = df[df["error"] == ""].reset_index(drop=True)
n_excluded = len(err_df)
n_analyzed = len(df)

if n_excluded:
    print(f"WARNING: {n_excluded} row(s) still have an error -- run the retry script first "
          f"for a complete analysis. Proceeding with the {n_analyzed} successful rows only.")

# ============================================================
# 1. Exact-match diagnosis
# ============================================================
PUNCT_RE = re.compile(r"['\u2018\u2019\u061B\u060C\u06D4۔]")  # quotes, Urdu comma/semicolon/full-stop

def strip_punct(s):
    return PUNCT_RE.sub("", s).strip()

df["gt_stripped"] = df["ground_truth"].apply(strip_punct)
df["pred_stripped"] = df["corrected_prediction"].apply(strip_punct)
exact_raw = (df["corrected_prediction"] == df["ground_truth"]).sum()
exact_stripped = (df["pred_stripped"] == df["gt_stripped"]).sum()

# ============================================================
# 2. CORRECTED repetition-loop detector (Section 27)
# ============================================================
def is_repetition_loop(text, min_repeats=3, coverage_thresh=0.40, ngram_range=(2, 6)):
    words = text.split()
    n_words = len(words)
    if n_words < min_repeats * 2:
        return False
    for n in range(ngram_range[0], min(ngram_range[1], n_words) + 1):
        ngrams = [tuple(words[i:i+n]) for i in range(n_words - n + 1)]
        counts = Counter(ngrams)
        for gram, cnt in counts.items():
            if cnt >= min_repeats:
                covered_tokens = cnt * n
                if covered_tokens / n_words >= coverage_thresh:
                    return True
    return False

df["is_repetition_loop"] = df["corrected_prediction"].apply(is_repetition_loop)

# --- Validation step, per Section 27's methodological note: run this BEFORE
# trusting the headline repetition-loop rate. Populate these lists manually
# from a worst-10 inspection pass (see section 3 below) before reporting a
# final number -- flagged here as a required manual step, not automated.
# known_loop_file_ids = [...]
# known_nonloop_file_ids = [...]

# ============================================================
# 3. Worst / best examples
# ============================================================
worst10 = df.nlargest(10, "cer_sensitive")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "cer_sensitive", "tesseract_cer_sensitive", "is_repetition_loop"]
]
best10_nonperfect = df[df["cer_sensitive"] > 0].nsmallest(10, "cer_sensitive")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "cer_sensitive", "tesseract_cer_sensitive"]
]

# ============================================================
# 4. Biggest gains / regressions vs Tesseract
# ============================================================
df["delta"] = df["tesseract_cer_sensitive"] - df["cer_sensitive"]
biggest_gains = df.nlargest(10, "delta")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "tesseract_cer_sensitive", "cer_sensitive", "delta"]
]
biggest_regressions = df.nsmallest(10, "delta")[
    ["file_id", "ground_truth", "tesseract_prediction", "corrected_prediction",
     "tesseract_cer_sensitive", "cer_sensitive", "delta", "is_repetition_loop"]
]

# ============================================================
# 5. Failure-mode categorization
# ============================================================
def categorize(row):
    pred, gt, tess = row["corrected_prediction"], row["ground_truth"], row["tesseract_prediction"]
    if pred == "":
        return "empty_output"
    if pred == tess:
        return "unchanged_copy_of_tesseract"
    if pred.strip() == gt.strip():
        return "exact_after_strip"
    if row["is_repetition_loop"]:
        return "repetition_loop"
    if row["cer_sensitive"] >= 1.5:
        return "fabrication_worse_than_tesseract_non_loop"
    if row["delta"] > 0.1:
        return "meaningful_improvement"
    if row["delta"] < -0.1:
        return "meaningful_regression"
    return "marginal_change"

df["failure_category"] = df.apply(categorize, axis=1)
category_counts = df["failure_category"].value_counts()

# ============================================================
# 6. Per-writer breakdown
# ============================================================
writer_stats = (
    df.groupby("writer_id")
    .agg(n=("cer_sensitive", "size"), mean_cer=("cer_sensitive", "mean"), mean_delta=("delta", "mean"))
    .query("n >= 5")
    .sort_values("mean_cer", ascending=False)
)

# ============================================================
# 7. Per-key breakdown -- Gemini-specific
# ============================================================
key_stats = None
if "key_used" in df.columns:
    key_stats = (
        df.groupby("key_used")
        .agg(n=("cer_sensitive", "size"), mean_cer=("cer_sensitive", "mean"))
        .sort_index()
    )

# ============================================================
# 8. Cross-model comparison -- three-way
# ============================================================
QWEN3VL_CONFIG3_CER = 0.4829
QWEN3VL_CONFIG3_LOOP_PCT = 1.34
QWEN25VL_CONFIG3_CER = 0.5984
QWEN25VL_CONFIG3_LOOP_PCT = 3.1
this_model_cer = df["cer_sensitive"].mean()
this_model_loop_pct = 100 * df["is_repetition_loop"].mean()

# ============================================================
# WRITE REPORT
# ============================================================
lines = []
lines.append("="*70)
lines.append("PART B -- CONFIG 3 (GEMINI 3.5 FLASH-LITE, IMAGE+TEXT, NO TEMPLATE CONTEXT)")
lines.append("QUALITATIVE ERROR ANALYSIS")
lines.append("="*70)

lines.append("\n--- 0. Coverage ---")
lines.append(f"Total rows in file: {n_total} | Excluded (still erroring): {n_excluded} | "
              f"Analyzed: {n_analyzed}")
if n_excluded:
    lines.append("--> Error rows excluded, not treated as CER=1.0 -- rerun the retry script "
                  "before treating this analysis as final.")

lines.append("\n--- 1. Exact-match diagnosis ---")
lines.append(f"Exact match, raw strings:            {exact_raw} / {n_analyzed} ({100*exact_raw/n_analyzed:.2f}%)")
lines.append(f"Exact match, punctuation-stripped:    {exact_stripped} / {n_analyzed} ({100*exact_stripped/n_analyzed:.2f}%)")
if exact_stripped > exact_raw:
    lines.append("--> Some 'failures' are punctuation-only mismatches, not content errors.")
else:
    lines.append("--> Exact-match rate is not inflated by punctuation stripping.")

lines.append("\n--- 2. Failure-mode category counts ---")
for cat, n in category_counts.items():
    lines.append(f"  {cat:38s}: {n:5d} ({100*n/n_analyzed:.1f}%)")

lines.append("\n--- 2b. REQUIRED MANUAL VALIDATION STEP (Section 27 discipline) ---")
lines.append("Manually inspect the worst-10 predictions below and confirm the repetition-loop")
lines.append("flag is behaving correctly on Gemini's outputs specifically -- don't assume the")
lines.append("Qwen-model validation transfers automatically.")

lines.append("\n--- 3. Worst 10 predictions (by corrected CER) ---")
for _, r in worst10.iterrows():
    lines.append(f"\n  [{r['file_id']}] corrected_CER={r['cer_sensitive']:.3f} | tesseract_CER={r['tesseract_cer_sensitive']:.3f} | flagged_loop={r['is_repetition_loop']}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 4. Best 10 non-perfect predictions ---")
for _, r in best10_nonperfect.iterrows():
    lines.append(f"\n  [{r['file_id']}] corrected_CER={r['cer_sensitive']:.3f} | tesseract_CER={r['tesseract_cer_sensitive']:.3f}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 5. Biggest 10 improvements over Tesseract ---")
for _, r in biggest_gains.iterrows():
    lines.append(f"\n  [{r['file_id']}] delta={r['delta']:+.3f}  (tess={r['tesseract_cer_sensitive']:.3f} -> corrected={r['cer_sensitive']:.3f})")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 6. Biggest 10 regressions vs Tesseract ---")
for _, r in biggest_regressions.iterrows():
    lines.append(f"\n  [{r['file_id']}] delta={r['delta']:+.3f}  (tess={r['tesseract_cer_sensitive']:.3f} -> corrected={r['cer_sensitive']:.3f}) | flagged_loop={r['is_repetition_loop']}")
    lines.append(f"    GT:        {r['ground_truth'][:100]}")
    lines.append(f"    Tesseract: {r['tesseract_prediction'][:100]}")
    lines.append(f"    Corrected: {r['corrected_prediction'][:100]}")

lines.append("\n--- 7. Per-writer CER (writers with >=5 samples, worst 10) ---")
lines.append(writer_stats.head(10).to_string())

if key_stats is not None:
    lines.append("\n--- 7b. Per-key CER (Gemini-specific) ---")
    lines.append(key_stats.to_string())

lines.append("\n--- 8. Cross-model comparison -- full three-way, all Config 3 correctors ---")
lines.append(f"  {'Corrector':<20}{'Part A CER':>14}{'Config 3 CER':>16}{'Loop %':>10}")
lines.append(f"  {'Qwen2.5-VL-7B':<20}{'0.6464':>14}{QWEN25VL_CONFIG3_CER:>16.4f}{QWEN25VL_CONFIG3_LOOP_PCT:>9.2f}%")
lines.append(f"  {'Qwen3-VL-8B':<20}{'0.4861':>14}{QWEN3VL_CONFIG3_CER:>16.4f}{QWEN3VL_CONFIG3_LOOP_PCT:>9.2f}%")
lines.append(f"  {'Gemini 3.5 Flash-Lite':<20}{'0.1718':>14}{this_model_cer:>16.4f}{this_model_loop_pct:>9.2f}%")
lines.append("")
lines.append("  Check: does this confirm the scaling pattern (weaker Part A corrector -> smaller")
lines.append("  Config 3 improvement, larger loop-dominated catastrophic tail) all the way down")
lines.append("  to Gemini's much stronger Part A baseline?")

report = "\n".join(lines)
print(report)
OUT_TXT.write_text(report, encoding="utf-8")
print(f"\n\nSaved full report to: {OUT_TXT}")

gemini_partB_config3_predictions.csv already present in /kaggle/working -- using existing copy.
PART B -- CONFIG 3 (GEMINI 3.5 FLASH-LITE, IMAGE+TEXT, NO TEMPLATE CONTEXT)
QUALITATIVE ERROR ANALYSIS

--- 0. Coverage ---
Total rows in file: 2465 | Excluded (still erroring): 0 | Analyzed: 2465

--- 1. Exact-match diagnosis ---
Exact match, raw strings:            212 / 2465 (8.60%)
Exact match, punctuation-stripped:    254 / 2465 (10.30%)
--> Some 'failures' are punctuation-only mismatches, not content errors.

--- 2. Failure-mode category counts ---
  meaningful_improvement                :  2118 (85.9%)
  exact_after_strip                     :   212 (8.6%)
  marginal_change                       :   116 (4.7%)
  meaningful_regression                 :    17 (0.7%)
  empty_output                          :     1 (0.0%)
  fabrication_worse_than_tesseract_non_loop:     1 (0.0%)

--- 2b. REQUIRED MANUAL VALIDATION STEP (Section 27 discipline) ---
Manually inspect the worst-10 predictions 